In [1]:
using Pkg
Pkg.activate(".")
# using Prime

  Activating project at `/home/mDisk/Prime-2`


In [2]:
using Distributed
addprocs(60)
@everywhere using Prime

In [7]:
using HDF5,DelimitedFiles,StatsBase
#Calculate cell-volume factor β
f = h5open("dataset/real_data/mouse_all.loom", "r")
X = read(f["layers/spliced"])+read(f["layers/unspliced"])
total_rd=sum(X,dims=2)
β = total_rd./mean(total_rd)

rdm = Matrix(readdlm("dataset/real_data/mouse_hvg_2020_spliced.csv",',')[2:end,2:end]')
rdn = Matrix(readdlm("dataset/real_data/mouse_hvg_2020_unspliced.csv", ',')[2:end,2:end]')

#cell number and gene number
n,p = size(rdm)
#Define cluster number and select model
k = 6
sel = 4

4

In [8]:
cluster_labels, γ = Prime.prime_cluster(
    rdn, rdm, β, k;
    sel = sel,
    maxiter = 20,  
)

┌ Info: PRIME iter 1, obj = 15.989998529240353
└ @ Prime /home/mDisk/Prime-2/src/cluster.jl:120
┌ Info: PRIME iter 2, obj = 15.905180123781204
└ @ Prime /home/mDisk/Prime-2/src/cluster.jl:120
┌ Info: PRIME iter 3, obj = 15.876702164364424
└ @ Prime /home/mDisk/Prime-2/src/cluster.jl:120
┌ Info: PRIME iter 4, obj = 15.893744437477963
└ @ Prime /home/mDisk/Prime-2/src/cluster.jl:120
┌ Info: PRIME iter 5, obj = 15.892760828666098
└ @ Prime /home/mDisk/Prime-2/src/cluster.jl:120
┌ Info: PRIME iter 6, obj = 15.753943171724451
└ @ Prime /home/mDisk/Prime-2/src/cluster.jl:120
┌ Info: PRIME iter 7, obj = 15.160766422703166
└ @ Prime /home/mDisk/Prime-2/src/cluster.jl:120
┌ Info: PRIME iter 8, obj = 13.886227532303167
└ @ Prime /home/mDisk/Prime-2/src/cluster.jl:120
┌ Info: PRIME iter 9, obj = 12.421753988112364
└ @ Prime /home/mDisk/Prime-2/src/cluster.jl:120
┌ Info: PRIME iter 10, obj = 11.434364499272748
└ @ Prime /home/mDisk/Prime-2/src/cluster.jl:120
┌ Info: PRIME iter 11, obj = 10.8501310

([1, 4, 1, 2, 3, 5, 1, 1, 2, 2  …  3, 6, 1, 3, 6, 4, 4, 1, 1, 3], [1.0315596085428855 7.295868025844947e-21 … 7.606591045567004e-9 1.1455666958428082e-11; 1.8779165533103354e-13 4.3093088421286786e-30 … 5.815717925129789e-13 1.4783334426140516e-15; … ; 1.0315596139281797 2.5635331875705637e-25 … 2.6175278808497434e-9 2.3277304677434067e-12; 0.3121955268787966 3.1526320711112757e-22 … 5.810282055355309e-7 1.5164098225915308e-6])

In [9]:
using Clustering
cell_type = readdlm("dataset/real_data/mouse_subclass2020_labels.csv",',')[2:end,2]
vec_numeric = replace(cell_type, 
"Lymphocytes"   => 1,
"Neutrophils"    => 2,
"Macrophages"   => 3,
"Small Airway Epithelial Cells"   => 4,
"Club Cells"     => 5,
"Endothelial Cells"     => 6,
)

println("ARI for PRIME clustering:",randindex(Int.(cluster_labels),Int.(vec_numeric))[1])
println("NMI for PRIME clustering:",mutualinfo(Int.(cluster_labels),Int.(vec_numeric)))

ARI for PRIME clustering:0.6827400689383778
NMI for PRIME clustering:0.7407887901410646
